# E-Commerce Sales Analysis — SQL Version

This notebook re-analyses the cleaned dataset using SQL (via DuckDB),
answering the same business questions as `E-Commerce_Sales_Analysis.ipynb`
using SQL instead of Pandas.

**Data source:** `data/data_clean.csv` (cleaned in the Python notebook)

## Setup

In [24]:
import duckdb
import pandas as pd

# Connect to DuckDB (in-memory database)
con = duckdb.connect()

# Load cleaned CSV as a table
con.execute("""
    CREATE TABLE sales AS 
    SELECT * FROM read_csv('data/data_clean.csv')
""")

# Verify row count matches the Python notebook
result = con.execute("SELECT COUNT(*) AS total_rows FROM sales").fetchdf()
print(result)

   total_rows
0      401560


## Query 1: Monthly Sales Trend
Aggregate total sales by month to identify seasonal trends,
matching the Python EDA results.

In [25]:
query = """
    SELECT 
        strftime(InvoiceDate, '%Y-%m') AS YearMonth,
        ROUND(SUM(TotalSales), 2) AS TotalSales
    FROM sales
    GROUP BY strftime(InvoiceDate, '%Y-%m')
    ORDER BY YearMonth
"""
monthly_sales_sql = con.execute(query).fetchdf()
print(monthly_sales_sql)

# Note: Results match the Python EDA (Nov 2011 peak at £1.13M, Dec 2011 partial)

   YearMonth  TotalSales
0    2010-12   552372.86
1    2011-01   473731.90
2    2011-02   435534.07
3    2011-03   578576.21
4    2011-04   425222.67
5    2011-05   647011.67
6    2011-06   606862.52
7    2011-07   573112.32
8    2011-08   615078.09
9    2011-09   929356.23
10   2011-10   973306.38
11   2011-11  1126815.07
12   2011-12   341539.43


## Query 2: Top 10 Products by Total Sales

Identify the best-selling products by total revenue, excluding
non-product items (postage, fees) and the incomplete Dec 2011 data.

In [26]:
query = """
    SELECT 
        Description,
        ROUND(SUM(TotalSales), 2) AS TotalSales
    FROM sales
    WHERE strftime(InvoiceDate, '%Y-%m') != '2011-12'
      AND Description NOT IN (
          'POSTAGE', 'CARRIAGE', 'MANUAL', 'DOTCOM POSTAGE',
          'BANK CHARGES', 'Manual', 'CRUK Commission', 'Discount'
      )
    GROUP BY Description
    ORDER BY TotalSales DESC
    LIMIT 10
"""

top_products_sql = con.execute(query).fetchdf()
print(top_products_sql)

                          Description  TotalSales
0            REGENCY CAKESTAND 3 TIER   128139.37
1  WHITE HANGING HEART T-LIGHT HOLDER    91774.96
2             JUMBO BAG RED RETROSPOT    81067.32
3                       PARTY BUNTING    67163.13
4       ASSORTED COLOUR BIRD ORNAMENT    54332.45
5                  RABBIT NIGHT LIGHT    44192.65
6                       CHILLI LIGHTS    43804.74
7      PICNIC BASKET WICKER 60 PIECES    39619.50
8     PAPER CHAIN KIT 50'S CHRISTMAS     37570.26
9             JUMBO BAG PINK POLKADOT    35635.36


## Query 3: Top 10 Countries by Total Sales

Rank markets by total revenue to identify the strongest sales regions.

In [27]:
query = """
    SELECT 
        Country,
        ROUND(SUM(TotalSales), 2) AS TotalSales
    FROM sales
    WHERE strftime(InvoiceDate, '%Y-%m') != '2011-12'
    GROUP BY Country
    ORDER BY TotalSales DESC
    LIMIT 10
"""

top_countries_sql = con.execute(query).fetchdf()
print(top_countries_sql)

          Country  TotalSales
0  United Kingdom  6450202.63
1     Netherlands   272933.52
2            EIRE   243025.41
3         Germany   213704.77
4          France   189577.22
5       Australia   137009.77
6     Switzerland    55739.40
7           Spain    54484.60
8         Belgium    39501.53
9          Sweden    36585.41


## Query 4: Top International Markets (Excluding UK)

Since the UK dominates sales, exclude it to reveal the relative
performance of international markets.

In [28]:
query = """
    SELECT 
        Country,
        ROUND(SUM(TotalSales), 2) AS TotalSales
    FROM sales
    WHERE strftime(InvoiceDate, '%Y-%m') != '2011-12'
      AND Country != 'United Kingdom'
    GROUP BY Country
    ORDER BY TotalSales DESC
    LIMIT 10
"""

top_intl_sql = con.execute(query).fetchdf()
print(top_intl_sql)

       Country  TotalSales
0  Netherlands   272933.52
1         EIRE   243025.41
2      Germany   213704.77
3       France   189577.22
4    Australia   137009.77
5  Switzerland    55739.40
6        Spain    54484.60
7      Belgium    39501.53
8       Sweden    36585.41
9        Japan    35457.55


## Query 5: Top 3 Products per Country (Window Function)

Use `ROW_NUMBER()` to rank products within each country and return
the top 3 per market. This demonstrates window functions and CTEs.

In [29]:
query = """
    WITH ranked_products AS (
        SELECT 
            Country,
            Description,
            ROUND(SUM(TotalSales), 2) AS TotalSales,
            ROW_NUMBER() OVER (
                PARTITION BY Country 
                ORDER BY SUM(TotalSales) DESC
            ) AS rank
        FROM sales
        WHERE strftime(InvoiceDate, '%Y-%m') != '2011-12'
          AND Description NOT IN (
              'POSTAGE', 'CARRIAGE', 'MANUAL', 'DOTCOM POSTAGE',
              'BANK CHARGES', 'Manual', 'CRUK Commission', 'Discount'
          )
        GROUP BY Country, Description
    )
    SELECT Country, Description, TotalSales, rank
    FROM ranked_products
    WHERE rank <= 3
    ORDER BY Country, rank
"""

top3_products_sql = con.execute(query).fetchdf()
print(top3_products_sql.head(15))

      Country                          Description  TotalSales  rank
0   Australia                   RABBIT NIGHT LIGHT     3375.84     1
1   Australia    SET OF 6 SPICE TINS PANTRY DESIGN     2082.00     2
2   Australia        RED TOADSTOOL LED NIGHT LIGHT     1987.20     3
3     Austria         PACK OF 6 SWEETIE GIFT BOXES      302.40     1
4     Austria       PACK OF 6 PANNETONE GIFT BOXES      302.40     2
5     Austria     RETROSPOT TEA SET CERAMIC 11 PC       205.95     3
6     Bahrain           ICE CREAM SUNDAE LIP GLOSS      120.00     1
7     Bahrain                  DOUGHNUT LIP GLOSS        75.00     2
8     Bahrain   NOVELTY BISCUITS CAKE STAND 3 TIER       59.70     3
9     Belgium  ROUND SNACK BOXES SET OF4 WOODLAND      1181.40     1
10    Belgium                 DOLLY GIRL LUNCH BOX      641.85     2
11    Belgium                  SPACEBOY LUNCH BOX       641.85     3
12     Brazil             REGENCY CAKESTAND 3 TIER      175.20     1
13     Brazil    SET OF 6 SPICE TI

## Query 6: Bottom 10 Countries by Total Sales

Identify the weakest markets by total revenue.

In [30]:
query = """
    SELECT 
        Country,
        ROUND(SUM(TotalSales), 2) AS TotalSales
    FROM sales
    WHERE strftime(InvoiceDate, '%Y-%m') != '2011-12'
    GROUP BY Country
    ORDER BY TotalSales ASC
    LIMIT 10
"""

bottom_countries_sql = con.execute(query).fetchdf()
print(bottom_countries_sql)

                Country  TotalSales
0          Saudi Arabia      131.17
1               Bahrain      548.40
2        Czech Republic      707.72
3                   RSA     1002.31
4                   USA     1115.64
5                Brazil     1143.60
6    European Community     1291.75
7             Lithuania     1661.06
8               Lebanon     1693.88
9  United Arab Emirates     1902.28


## Query 7: Bottom 3 Products per Country (Window Function)

Use `ROW_NUMBER()` to find the lowest-selling products within each
market, considering only products with positive sales.

In [31]:
query = """
    WITH ranked_products AS (
        SELECT 
            Country,
            Description,
            ROUND(SUM(TotalSales), 2) AS TotalSales,
            ROW_NUMBER() OVER (
                PARTITION BY Country 
                ORDER BY SUM(TotalSales) ASC
            ) AS rank
        FROM sales
        WHERE strftime(InvoiceDate, '%Y-%m') != '2011-12'
          AND Description NOT IN (
              'POSTAGE', 'CARRIAGE', 'MANUAL', 'DOTCOM POSTAGE',
              'BANK CHARGES', 'Manual', 'CRUK Commission', 'Discount'
          )
        GROUP BY Country, Description
        HAVING SUM(TotalSales) > 0
    )
    SELECT Country, Description, TotalSales, rank
    FROM ranked_products
    WHERE rank <= 3
    ORDER BY Country, rank
"""

bottom3_products_sql = con.execute(query).fetchdf()
print(bottom3_products_sql.head(15))

      Country                         Description  TotalSales  rank
0   Australia         PASTEL COLOUR HONEYCOMB FAN        4.68     1
1   Australia           ELEPHANT, BIRTHDAY CARD,         5.04     2
2   Australia                 CARD BILLBOARD FONT        5.04     3
3     Austria                     BLUE EGG  SPOON        2.88     1
4     Austria                     RED  EGG  SPOON        2.88     2
5     Austria        PENNY FARTHING BIRTHDAY CARD        5.04     3
6     Bahrain  MINI CAKE STAND WITH HANGING CAKES       11.60     1
7     Bahrain   CERAMIC CAKE BOWL + HANGING CAKES       17.70     2
8     Bahrain      PINK REGENCY TEACUP AND SAUCER       17.70     3
9     Belgium      POLYESTER FILLER PAD 30CMx30CM        2.50     1
10    Belgium         ICON PLACEMAT POP ART ELVIS        2.88     2
11    Belgium     DISCO BALL CHRISTMAS DECORATION        2.88     3
12     Brazil            EMERGENCY FIRST AID TIN        15.00     1
13     Brazil          RECYCLED ACAPULCO MAT PIN

# Summary

This SQL analysis reproduces the key findings from the Python EDA,
demonstrating data aggregation, filtering, and window functions
using DuckDB.